# NHANES Adult Body Measurements
#Done by Muhammed Rushwan Rasheed
#Internship project

## Capstone Project 1 - Working with NumPy Matrices

**Purpose.** This report analyses adult male and female body measurements from the NHANES excerpts supplied in the project brief. The work uses NumPy matrices as the central data structure and develops comparable graphical and numerical summaries.

The source files contain seven measurements: weight, height, upper-arm length, upper-leg length, arm circumference, hip circumference, and waist circumference. All lengths are in centimetres and weight is in kilograms.

## 1. Setup and data import

The next cell imports the libraries used throughout the report and reads the two CSV files into NumPy matrices. Comment lines in the source files are ignored. A short validation confirms the expected seven original columns.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=3, suppress=True)

# Supports running from either the project root or the outputs folder.
DATA_DIR = Path('data')
if not DATA_DIR.exists():
    DATA_DIR = Path('../data')
COLUMN_NAMES = np.array([
    'Weight (kg)', 'Height (cm)', 'Upper arm (cm)', 'Upper leg (cm)',
    'Arm circumference (cm)', 'Hip circumference (cm)', 'Waist circumference (cm)'
])

# The first 18 rows are comments and row 19 is the source-file header.
male = np.loadtxt(DATA_DIR / 'nhanes_adult_male_bmx_2020.csv', delimiter=',', comments='#', skiprows=19)
female = np.loadtxt(DATA_DIR / 'nhanes_adult_female_bmx_2020.csv', delimiter=',', comments='#', skiprows=19)

print(f'Male matrix shape:   {male.shape}')
print(f'Female matrix shape: {female.shape}')
print(f'Original columns: {len(COLUMN_NAMES)}')
print('Missing values:', np.isnan(male).sum() + np.isnan(female).sum())
assert male.shape[1] == female.shape[1] == 7


The two matrices have the required seven columns. All subsequent calculations retain this matrix-based workflow; derived measures are appended as additional columns where requested.

## 2. Weight distributions

This figure uses two vertically stacked histograms to compare female and male weights. Identical x-axis limits make differences in centre, spread, and tail behaviour visually comparable.

In [ ]:
female_weights, male_weights = female[:, 0], male[:, 0]
xmin = 5 * np.floor(min(female_weights.min(), male_weights.min()) / 5)
xmax = 5 * np.ceil(max(female_weights.max(), male_weights.max()) / 5)
bins = np.arange(xmin, xmax + 5, 5)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True, constrained_layout=True)
axes[0].hist(female_weights, bins=bins, color='#c65b7c', edgecolor='white')
axes[0].set(title='Female weight distribution', ylabel='Number of participants')
axes[1].hist(male_weights, bins=bins, color='#3b82b5', edgecolor='white')
axes[1].set(title='Male weight distribution', xlabel='Weight (kg)', ylabel='Number of participants')
for ax in axes:
    ax.set_xlim(xmin, xmax)
plt.show()
print(f'Common x-axis limits: {xmin:.0f} to {xmax:.0f} kg')


Both distributions can be inspected on the same horizontal scale. The common limits prevent a misleading comparison caused by automatic rescaling and make any right tail especially easy to see.

## 3. Boxplot comparison of weight

The boxplot gives a compact comparison of the medians, interquartile ranges, whiskers, and potential outliers for female and male weights.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([female_weights, male_weights], tick_labels=['Female', 'Male'], patch_artist=True,
           boxprops=dict(facecolor='#d9eaf7'), medianprops=dict(color='#b22222', linewidth=2))
ax.set(title='Comparison of adult body weight', ylabel='Weight (kg)')
plt.show()

for label, values in [('Female', female_weights), ('Male', male_weights)]:
    q1, median, q3 = np.percentile(values, [25, 50, 75])
    print(f'{label}: Q1={q1:.1f}, median={median:.1f}, Q3={q3:.1f}, IQR={q3-q1:.1f} kg')


The relative vertical position of the boxes shows which group has the higher typical weight. Box heights compare middle-half variation, while points beyond the whiskers flag unusually low or high observations under the conventional 1.5 IQR rule.

## 4. Numerical aggregates and distribution shape

The following function calculates measures of location, dispersion, and shape. Standard deviation and variance are reported as sample statistics (`ddof=1`); skewness and excess kurtosis use moment-based estimates.

In [ ]:
def descriptive_statistics(x):
    n = x.size
    mean = x.mean()
    centred = x - mean
    s = x.std(ddof=1)
    skewness = n / ((n - 1) * (n - 2)) * np.sum((centred / s) ** 3)
    excess_kurtosis = (n * (n + 1) / ((n - 1) * (n - 2) * (n - 3))
                       * np.sum((centred / s) ** 4)
                       - 3 * (n - 1) ** 2 / ((n - 2) * (n - 3)))
    return {
        'n': n, 'mean': mean, 'median': np.median(x), 'minimum': x.min(), 'maximum': x.max(),
        'range': np.ptp(x), 'variance': x.var(ddof=1), 'std. deviation': s,
        'IQR': np.diff(np.percentile(x, [25, 75]))[0], 'skewness': skewness,
        'excess kurtosis': excess_kurtosis
    }

female_stats, male_stats = descriptive_statistics(female_weights), descriptive_statistics(male_weights)
print(f"{'Statistic':<18}{'Female':>12}{'Male':>12}")
for key in female_stats:
    print(f"{key:<18}{female_stats[key]:>12.3f}{male_stats[key]:>12.3f}")

for label, stats in [('Female', female_stats), ('Male', male_stats)]:
    direction = 'right-skewed' if stats['skewness'] > 0.1 else ('left-skewed' if stats['skewness'] < -0.1 else 'approximately symmetric')
    print(f'{label} weights are {direction} (skewness = {stats["skewness"]:.2f}).')
print(('Female' if female_stats['std. deviation'] > male_stats['std. deviation'] else 'Male') + ' weights have the larger sample standard deviation.')


The location measures describe a typical participant, whereas standard deviation, IQR, and range describe dispersion. A positive skewness indicates a longer upper-weight tail; excess kurtosis above zero indicates heavier tails than a normal distribution. The printed comparison identifies the more dispersed sample directly from the data.

## 5. BMI and female z-scores

BMI is calculated as weight divided by squared height in metres. It is appended as the eighth column of the female matrix. Then every female column, including BMI, is standardised to a z-score using its sample mean and sample standard deviation.

In [ ]:
female_bmi = female[:, 0] / (female[:, 1] / 100) ** 2
female = np.column_stack((female, female_bmi))
female_column_names = np.append(COLUMN_NAMES, 'BMI (kg/m²)')

female_means = female.mean(axis=0)
female_stds = female.std(axis=0, ddof=1)
zfemale = (female - female_means) / female_stds

print('Female matrix after BMI:', female.shape)
print('Maximum absolute z-score mean:', np.abs(zfemale.mean(axis=0)).max())
print('Sample standard deviations of zfemale:', zfemale.std(axis=0, ddof=1).round(3))


The augmented female matrix now has eight columns. Standardisation puts all variables on the same scale: a z-score of +1 means one sample standard deviation above that variable’s mean. The numerical checks confirm that each standardised column has mean near zero and sample standard deviation one.

## 6. Female scatterplot matrix and correlations

This scatterplot matrix uses standardised height, weight, waist circumference, hip circumference, and BMI. The diagonal shows distributions; off-diagonal panels show pairwise relationships. Pearson correlation measures linear association, while Spearman correlation measures monotonic association based on ranks.

In [ ]:
selected_idx = np.array([1, 0, 6, 5, 7])
selected_names = female_column_names[selected_idx]
selected_z = zfemale[:, selected_idx]

fig, axes = plt.subplots(5, 5, figsize=(12, 12), constrained_layout=True)
for i in range(5):
    for j in range(5):
        ax = axes[i, j]
        if i == j:
            ax.hist(selected_z[:, i], bins=18, color='#7aa6c2', edgecolor='white')
        else:
            ax.scatter(selected_z[:, j], selected_z[:, i], s=10, alpha=0.45, color='#315a7d')
        if i == 4: ax.set_xlabel(selected_names[j], rotation=35, ha='right')
        else: ax.set_xticks([])
        if j == 0: ax.set_ylabel(selected_names[i])
        else: ax.set_yticks([])
fig.suptitle('Female standardised body measurements', fontsize=16)
plt.show()

def rankdata(a):
    order = np.argsort(a, kind='mergesort')
    ranks = np.empty(a.size, float)
    sorted_a = a[order]
    start = 0
    while start < a.size:
        end = start + 1
        while end < a.size and sorted_a[end] == sorted_a[start]: end += 1
        ranks[order[start:end]] = (start + end - 1) / 2 + 1
        start = end
    return ranks

pearson = np.corrcoef(selected_z, rowvar=False)
spearman = np.corrcoef(np.column_stack([rankdata(selected_z[:, i]) for i in range(5)]), rowvar=False)
print('Pearson correlation matrix (rows/columns in this order):', list(selected_names))
print(np.round(pearson, 3))
print('\nSpearman correlation matrix (same order):')
print(np.round(spearman, 3))


Strong positive coefficients indicate that the two measurements tend to increase together; coefficients near zero indicate little linear or monotonic association. Comparing Pearson and Spearman values helps distinguish a broadly monotonic relationship from one that is specifically linear. The clearest relationships can be identified by the largest absolute off-diagonal coefficients printed above.

## 7. Waist ratios for males and females

Waist-to-height ratio and waist-to-hip ratio are appended to both matrices. These dimensionless ratios adjust waist size for stature and hip size, respectively.

In [ ]:
# Existing columns: height=1, hip=5, waist=6
def add_waist_ratios(x):
    waist_to_height = x[:, 6] / x[:, 1]
    waist_to_hip = x[:, 6] / x[:, 5]
    return np.column_stack((x, waist_to_height, waist_to_hip))

male = add_waist_ratios(male)
female = add_waist_ratios(female)
ratio_names = ['Waist / height', 'Waist / hip']
print('Male matrix after ratios:', male.shape)
print('Female matrix after ratios:', female.shape)
print('Female final columns:', list(female_column_names) + ratio_names)


The male matrix now has nine columns (seven original measurements plus two ratios). The female matrix has ten columns because it additionally contains BMI. The ratio columns are ready for direct comparison.

## 8. Comparison of waist ratios

The four-box plot compares female and male waist-to-height ratios and waist-to-hip ratios side by side. Medians and IQRs make group differences visible without assuming normal distributions.

In [ ]:
f_wth, f_wthip = female[:, -2], female[:, -1]
m_wth, m_wthip = male[:, -2], male[:, -1]
fig, ax = plt.subplots(figsize=(11, 5))
boxes = ax.boxplot([f_wth, m_wth, f_wthip, m_wthip], patch_artist=True,
                   tick_labels=['Female\nwaist/height', 'Male\nwaist/height',
                                'Female\nwaist/hip', 'Male\nwaist/hip'])
for patch, color in zip(boxes['boxes'], ['#d9819a', '#6aa2c8', '#d9819a', '#6aa2c8']): patch.set_facecolor(color)
ax.set(title='Waist ratio distributions by sex', ylabel='Ratio')
plt.show()

for label, x in [('Female waist/height', f_wth), ('Male waist/height', m_wth),
                 ('Female waist/hip', f_wthip), ('Male waist/hip', m_wthip)]:
    print(f'{label:<24} median={np.median(x):.3f}, IQR={np.diff(np.percentile(x,[25,75]))[0]:.3f}')


The boxplot compares central ratio values and their variability across the four distributions. Any separation of medians suggests a systematic group difference, while differing box heights indicate different variability. Outlying points represent participants with unusually low or high ratios relative to their group.

## 9. Interpreting BMI and waist-based ratios

No single anthropometric measure is definitive. Their strengths and limitations should be considered together rather than treating one metric as a diagnosis.

| Measure | Advantages | Limitations |
|---|---|---|
| **BMI** | Simple, inexpensive, standardised, and useful for population-level screening. | Does not separate muscle from fat or describe fat distribution; thresholds can be less informative across individuals and populations. |
| **Waist-to-height ratio** | Incorporates central size and stature; easy to calculate and often intuitive for comparing people of different heights. | Measurement location and posture affect waist values; it remains a screening measure, not a direct measure of adiposity. |
| **Waist-to-hip ratio** | Captures body-fat distribution and is independent of overall body size to some extent. | Depends on two measurement sites, can be less stable when hip size reflects muscularity or skeletal structure, and is sensitive to measurement error. |

## 10. Lowest and highest female BMI observations

The final analysis selects five women with the lowest and five with the highest BMI using `numpy.argsort`. Their standardised measurements are printed so that values are comparable across variables.

In [ ]:
bmi_index = 7
order = np.argsort(female[:, bmi_index])
chosen = np.concatenate((order[:5], order[-5:]))
print('Rows are ordered as five lowest BMI, then five highest BMI.')
print('Columns:', list(female_column_names))
print(zfemale[chosen])

print('\nBMI values for the selected participants:')
print(np.round(female[chosen, bmi_index], 2))
print('\nInterpretation: negative z-scores are below the female sample mean; positive z-scores are above it. The BMI column is expected to be strongly negative for the first five rows and strongly positive for the final five rows. Comparing waist, hip, and weight z-scores shows which measurements most accompany these extremes.')


## Conclusion

This notebook imports the NHANES excerpts as NumPy matrices, compares the weight distributions graphically and numerically, derives BMI and waist ratios, standardises female measurements, and evaluates pairwise associations. The results should be interpreted as descriptive summaries of these excerpts, not as clinical diagnoses. In practical health assessment, anthropometric screening measures are best considered alongside medical history, examination, and other relevant evidence.